# StyleGAN2-ADA Approach
- Ref paper: https://arxiv.org/pdf/2203.13856
- Paper summary: This paper addresses the challenge of small and imbalanced medical datasets by leveraging synthetic data augmentation to improve Age-related Macular Degeneration (AMD) classification. After evaluating ten different Generative Adversarial Networks, the researchers identified StyleGAN2 as the superior architecture, achieving the lowest FID scores and generating synthetic fundus images realistic enough to fool clinical experts in a visual Turing test. Ultimately, augmenting the training set with these high-quality synthetic images significantly boosted the performance of a downstream ResNet-18 classifier.

# Set Up

In [2]:
!nvidia-smi -L

from google.colab import drive
drive.mount('/content/drive')

GPU 0: NVIDIA L4 (UUID: GPU-bae0762b-d9e6-a033-d049-15d081a22de5)
Mounted at /content/drive


In [2]:
# Install Python 3.8
# Colab runs Python 3.12 — SG3 custom CUDA ops need Python 3.8
# This installs 3.8 alongside the system Python and switches to it

!sudo apt-get update -y -q
!sudo apt-get install -y -q python3.8 python3.8-distutils

# Register 3.8 as an alternative and auto-select it (priority 2 > default 1)
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.8 2
!sudo update-alternatives --set python3 /usr/bin/python3.8

# Verify
!python3 --version   # should show Python 3.8.x

# Fix pip for 3.8
!apt-get install -y -q python3-pip
!python3 -m pip install --upgrade pip --quiet

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backpo

In [3]:
# Install dependencies

# Remove incompatible versions
!pip uninstall -y torch torchvision torchaudio jax jaxlib 2>/dev/null

# PyTorch 1.9.0 + CUDA 11.1 — exact version SG3 custom ops compile against
!pip install -q torch==1.9.0+cu111 torchvision==0.10.0+cu111 \
    -f https://download.pytorch.org/whl/torch_stable.html

# Pin all other deps to match reference notebook exactly
!pip install -q \
    numpy==1.20.0 \
    click==8.0 \
    pillow==8.3.1 \
    scipy==1.7.1 \
    requests==2.26.0 \
    tqdm==4.62.2 \
    matplotlib==3.4.2 \
    imageio==2.9.0 \
    imageio-ffmpeg==0.4.3 \
    ninja==1.10.2 \
    timm==0.4.12 \
    ftfy==6.1.1 \
    opensimplex \
    pyspng \
    psutil \
    pandas \
    clean-fid \
    pytorch-fid

# Verify
import torch
print(f"PyTorch : {torch.__version__}")    # 1.9.0+cu111
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 221.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 163.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 153.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.6 MB/s eta 0:00:00
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVI

In [4]:
# Clone dvschultz StyleGAN2-ADA
import os

SG2_DIR = "/content/stylegan2_ada"
if not os.path.isdir(SG2_DIR):
    !git clone https://github.com/dvschultz/stylegan2-ada-pytorch.git {SG2_DIR}
    print("Cloned StyleGAN2-ADA ✓")
else:
    print("Already cloned ✓")

os.chdir(SG2_DIR)

# Verify it works
!python train.py --help

Cloning into '/content/stylegan2_ada'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 577 (delta 16), reused 12 (delta 12), pack-reused 552 (from 2)
Receiving objects: 100% (577/577), 8.43 MiB | 19.48 MiB/s, done.
Resolving deltas: 100% (329/329), done.
Cloned StyleGAN2-ADA ✓
Usage: train.py [OPTIONS]

  Train a GAN using the techniques described in
  the paper "Training Generative Adversarial
  Networks with Limited Data".

  Examples:

  # Train with custom dataset using 1 GPU.
  python train.py --outdir=~/training-runs --data=~/mydataset.zip --gpus=1

  # Train class-conditional CIFAR-10 using 2 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/cifar10.zip \
      --gpus=2 --cfg=cifar --cond=1

  # Transfer learn MetFaces from FFHQ using 4 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/metfaces.zip \
      --gpus=4 --cfg=paper1024 --mirror=1 --

In [5]:
# Transfer dataset from Drive to local SSD
import subprocess
def run(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True)

ARCHIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/archive.zip"
run(f"cp '{ARCHIVE_PATH}' /content/RFMiD.zip")
run("unzip -q /content/RFMiD.zip -d /content/RFMiD")
run("rm -rf /content/RFMiD/__MACOSX")
run('find /content/RFMiD -name ".DS_Store" -delete')

print("Dataset ready ✓")

# Verify train.py works before going further
os.chdir(SG2_DIR)
!python train.py --help

Dataset ready ✓
Usage: train.py [OPTIONS]

  Train a GAN using the techniques described in
  the paper "Training Generative Adversarial
  Networks with Limited Data".

  Examples:

  # Train with custom dataset using 1 GPU.
  python train.py --outdir=~/training-runs --data=~/mydataset.zip --gpus=1

  # Train class-conditional CIFAR-10 using 2 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/cifar10.zip \
      --gpus=2 --cfg=cifar --cond=1

  # Transfer learn MetFaces from FFHQ using 4 GPUs.
  python train.py --outdir=~/training-runs --data=~/datasets/metfaces.zip \
      --gpus=4 --cfg=paper1024 --mirror=1 --resume=ffhq1024 --snap=10

  # Reproduce original StyleGAN2 config F.
  python train.py --outdir=~/training-runs --data=~/datasets/ffhq.zip \
      --gpus=8 --cfg=stylegan2 --mirror=1 --aug=noaug

  Base configs (--cfg):
    auto       Automatically select reasonable defaults based on resolution
               and GPU count. Good starting point for new datasets.


# Label mapping

- We selected Retinal Fundus Multi-disease image Dataset (RFMiD) for experimentation in this project.
- Class Distribution of normal 669 images and diseased of 2531 images. (Count by 'Disease_Risk' column), but since the labels and images are separate from one another, we perform the mapping first.
- The data has already been resized to 512 x 512 and center cropped.

In [ ]:
import os, shutil, pandas as pd
from pathlib import Path
from tqdm import tqdm

BASE         = "/content/RFMiD/archive"
CSV_TRAIN    = f"{BASE}/Training_Set/RFMiD_Training_Labels.csv"
CSV_VAL      = f"{BASE}/Evaluation_Set/RFMiD_Validation_Labels.csv"
CSV_TEST     = f"{BASE}/Test_Set/RFMiD_Testing_Labels.csv"
IMG_TRAIN    = f"{BASE}/Training_Set/Training"
IMG_VAL      = f"{BASE}/Evaluation_Set/Validation"
IMG_TEST     = f"{BASE}/Test_Set/Test"

NORMAL_RAW   = "/content/data/normal"
ABNORMAL_RAW = "/content/data/abnormal"

os.makedirs(NORMAL_RAW,   exist_ok=True)
os.makedirs(ABNORMAL_RAW, exist_ok=True)

EXTENSIONS = [".png", ".jpg", ".jpeg", ".PNG", ".JPG"]

def resolve(img_id, img_dir):
    for ext in EXTENSIONS:
        p = os.path.join(img_dir, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return None

def load_split(csv_path, img_dir, split_name):
    if not os.path.exists(csv_path):
        print(f"  CSV not found: {csv_path}")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    df["img_path"] = df["ID"].apply(lambda x: resolve(x, img_dir))
    df["split"]    = split_name
    found = df["img_path"].notna().sum()
    print(f"  {split_name}: {len(df)} rows | {found} images found | "
          f"{len(df)-found} missing")
    return df

print("Loading splits...")
train_df = load_split(CSV_TRAIN, IMG_TRAIN, "train")
val_df   = load_split(CSV_VAL,   IMG_VAL,   "val")
test_df  = load_split(CSV_TEST,  IMG_TEST,  "test")

all_df      = pd.concat([train_df, val_df, test_df], ignore_index=True)
normal_df   = all_df[(all_df["Disease_Risk"] == 0) & all_df["img_path"].notna()].reset_index(drop=True)
abnormal_df = all_df[(all_df["Disease_Risk"] == 1) & all_df["img_path"].notna()].reset_index(drop=True)

print(f"\nNormal   : {len(normal_df)}")
print(f"Abnormal : {len(abnormal_df)}")

# Copy with unique filenames
def copy_class(df, out_dir, label):
    copied = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {label}"):
        orig_name       = os.path.basename(row["img_path"])
        unique_filename = f"{row['split']}_{orig_name}"   # e.g. train_123.png
        dst             = os.path.join(out_dir, unique_filename)
        if not os.path.exists(dst):
            shutil.copy2(row["img_path"], dst)
        copied += 1

    actual = len(list(Path(out_dir).iterdir()))
    print(f"  {label}: {actual} images in folder")
    return actual

n_normal   = copy_class(normal_df,   NORMAL_RAW,   "Normal")
n_abnormal = copy_class(abnormal_df, ABNORMAL_RAW, "Abnormal")

In [ ]:
import json, shutil, os
from pathlib import Path
from tqdm import tqdm

COMBINED_RAW = "/content/data/combined"
COMBINED_ZIP = "/content/sg2ada_combined.zip"
RESOLUTION   = 256

os.makedirs(COMBINED_RAW, exist_ok=True)

labels = []

print("Building combined labelled folder...")

# Define the reusable processing function
def process_images(df, prefix, class_label, description):
    print(f"Processing {description}...")
    for i, row in tqdm(df.iterrows(), total=len(df), desc=description):
        # Using the index 'i' guarantees uniqueness and prevents collisions
        fname = f"{prefix}_{i}_{os.path.basename(row['img_path'])}"
        dst = os.path.join(COMBINED_RAW, fname)

        shutil.copy2(row["img_path"], dst)
        labels.append([fname, class_label])

# Run the function for each dataframe
process_images(normal_df, prefix="n", class_label=0, description="Normal")
process_images(abnormal_df, prefix="a", class_label=1, description="Abnormal")

# Write dataset.json — SG2-ADA reads this for class labels
with open(os.path.join(COMBINED_RAW, "dataset.json"), "w") as f:
    json.dump({"labels": labels}, f)

n_normal_combined   = sum(1 for _, l in labels if l == 0)
n_abnormal_combined = sum(1 for _, l in labels if l == 1)

print(f"\nCombined folder : {len(labels)} images")
print(f"  Normal        : {n_normal_combined}")
print(f"  Abnormal      : {n_abnormal_combined}")
print(f"  Label file    : {COMBINED_RAW}/dataset.json ✓")

# Convert to SG2-ADA dataset zip
os.chdir("/content/stylegan2_ada")

!python dataset_tool.py \
    --source={COMBINED_RAW} \
    --dest={COMBINED_ZIP} \
    --width={RESOLUTION} \
    --height={RESOLUTION}

print(f"\nCombined dataset zip: {COMBINED_ZIP}")

- To train StyleGAN2-ADA conditionally, we must first combine all images into a single directory and map their classes (Normal/Abnormal) using a dataset.json file. The dataset_tool.py script is then required to package these raw images and labels into a highly optimized, uncompressed .zip archive. This essential preprocessing step standardizes all images to an exact square resolution (256x256).

# Config

- resolution = 256: Generates 256×256 images.
- cond = True: Enables conditional training. Instead of generating random eye images, this forces the model to learn the specific differences between classes. This is exactly how the paper balances the dataset.
- total_kimg: 1000: GAN training length is measured in "kimg" (thousands of real images shown to the discriminator). Setting this to 1000 means the model will train until it has processed 1,000,000 images.

ADA (Adaptive Discriminator Augmentation)
This is the most critical block for medical imaging, addressing the paper's core problem (small, limited datasets).

- aug: "ada" & ada_target: 0.6: When training on small datasets, the discriminator quickly memorizes the real images and stops giving useful feedback. ADA dynamically adjusts the strength of data augmentations to keep the discriminator's confidence around the target of 60% (0.6). This prevents overfitting.

In [ ]:
from dataclasses import dataclass

@dataclass
class SG2ADACondConfig:
    resolution  : int   = 256
    batch       : int   = 32
    batch_gpu   : int   = 4
    total_kimg  : int   = 1000
    snap        : int   = 10
    aug         : str   = "ada"
    augpipe     : str   = "bgcfnc"
    ada_target  : float = 0.6
    cond        : bool  = True

    combined_zip : str  = "/content/sg2ada_combined.zip"
    outdir       : str  = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/sg2ada_conditional"

cfg = SG2ADACondConfig()
os.makedirs(cfg.outdir, exist_ok=True)

print("Config ready ✓")
print(f"  Conditional : {cfg.cond}")
print(f"  Classes     : 0=normal  1=abnormal")
print(f"  Aug         : {cfg.aug} / {cfg.augpipe}")

In [9]:
# Fix: Install Python 3.8 dev headers
!sudo apt-get install -y python3.8-dev

# Verify Python.h now exists
!find /usr/include -name "Python.h" 2>/dev/null
# Should show: /usr/include/python3.8/Python.h

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.8 libpython3.8-dev
The following NEW packages will be installed:
  libpython3.8 libpython3.8-dev python3.8-dev
0 upgraded, 3 newly installed, 0 to remove and 58 not upgraded.
Need to get 6,687 kB of archives.
After this operation, 25.0 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.8 amd64 3.8.20-1+jammy1 [1,798 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.8-dev amd64 3.8.20-1+jammy1 [4,389 kB]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 python3.8-dev amd64 3.8.20-1+jammy1 [500 kB]
Fetched 6,687 kB in 5s (1,477 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at

# Training

- We clone the official StyleGAN2-ADA GitHub repository directly into our Google Colab environment and leverage their built-in train.py script, this single script trains one model capable of generating both "Normal" and "Abnormal" classes.

In [10]:
# # Smoke test
# os.chdir("/content/stylegan2_ada")

# !python train.py \
#     --outdir=/content/smoke_test_sg2ada_cond \
#     --data={cfg.combined_zip} \
#     --gpus=1 \
#     --batch={cfg.batch} \
#     --cfg=auto \
#     --cond=True \
#     --aug={cfg.aug} \
#     --augpipe={cfg.augpipe} \
#     --target={cfg.ada_target} \
#     --kimg=1 \
#     --snap=1 \
#     --metrics=none \
#     --mirror=1

# import glob
# snaps = glob.glob("/content/smoke_test_sg2ada_cond/**/*.pkl", recursive=True)
# print(f"\nSmoke test {'PASSED ✓' if snaps else 'FAILED ✗'}")

In [17]:
# Single training run — both classes
import glob
os.chdir("/content/stylegan2_ada")

def get_latest_pkl(run_dir):
    pkls = sorted(glob.glob(
        os.path.join(run_dir, "**", "network-snapshot-*.pkl"),
        recursive=True
    ))
    return pkls[-1] if pkls else None

resume      = get_latest_pkl(cfg.outdir)
resume_flag = f"--resume={resume}" if resume else ""
print(f"{'Resuming: ' + resume if resume else 'Starting from scratch'}")

!python train.py \
    --outdir="{cfg.outdir}" \
    --data="{cfg.combined_zip}" \
    --gpus=1 \
    --batch={cfg.batch} \
    --cfg=auto \
    --cond=True \
    --aug={cfg.aug} \
    --augpipe={cfg.augpipe} \
    --target={cfg.ada_target} \
    --kimg={cfg.total_kimg} \
    --snap={cfg.snap} \
    --metrics=none \
    --mirror=1 \
    {resume_flag}

Starting from scratch

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 10,
  "network_snapshot_ticks": 10,
  "metrics": [],
  "random_seed": 0,
  "training_set_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "/content/sg2ada_combined.zip",
    "use_labels": true,
    "max_size": 3200,
    "xflip": true,
    "resolution": 256
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 3,
    "prefetch_factor": 2
  },
  "G_kwargs": {
    "class_name": "training.networks.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 2
    },
    "synthesis_kwargs": {
      "channel_base": 16384,
      "channel_max": 512,
      "num_fp16_res": 4,
      "conv_clamp": 256
    }
  },
  "D_kwargs": {
    "class_name": "training.networks.Discriminator",
    "block_kwargs": {},
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 16384,
    "channel_max":

- last training tick 250, kimg 1000.0, time 3h 30m 46s

# Generate Synthetic Images

- We will generate a normal and abnormal class with each 500 samples, totaling 1,000 samples.

In [18]:
import os
import glob
from pathlib import Path

os.chdir("/content/stylegan2_ada")

GEN_NORMAL_DIR   = "/content/generated/sg2ada_cond_normal"
GEN_ABNORMAL_DIR = "/content/generated/sg2ada_cond_abnormal"
N_GENERATE       = 1000
TRUNCATION       = 0.7

os.makedirs(GEN_NORMAL_DIR,   exist_ok=True)
os.makedirs(GEN_ABNORMAL_DIR, exist_ok=True)

best_pkl = get_latest_pkl(cfg.outdir)

if best_pkl:
    print(f"Using: {best_pkl}\n")

    # Generate normal (class 0)
    print("Generating normal class (class=0)...")
    !python generate.py \
        --outdir="{GEN_NORMAL_DIR}" \
        --trunc={TRUNCATION} \
        --seeds=0-{N_GENERATE - 1} \
        --class=0 \
        --network="{best_pkl}"

    # Generate abnormal (class 1)
    print("\nGenerating abnormal class (class=1)...")
    !python generate.py \
        --outdir="{GEN_ABNORMAL_DIR}" \
        --trunc={TRUNCATION} \
        --seeds=0-{N_GENERATE - 1} \
        --class=1 \
        --network="{best_pkl}"

    print(f"\nGenerated normal   : {len(list(Path(GEN_NORMAL_DIR).glob('*.png')))}")
    print(f"Generated abnormal : {len(list(Path(GEN_ABNORMAL_DIR).glob('*.png')))}")
else:
    print("No checkpoint found — train first")

Using: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/sg2ada_conditional/00001-sg2ada_combined-cond-mirror-auto1-kimg1000-batch32-ada-target0.6-bgcfnc/network-snapshot-001000.pkl

Generating normal class (class=0)...
generate.py:59: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  elif(len(seeds) is not 3):
Loading networks from "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/sg2ada_conditional/00001-sg2ada_combined-cond-mirror-auto1-kimg1000-batch32-ada-target0.6-bgcfnc/network-snapshot-001000.pkl"...
Generating image for seed 0 (0/1000) ...
Setting up PyTorch plugin "bias_act_plugin"... Done.
Setting up PyTorch plugin "upfirdn2d_plugin"... Done.
Generating image for seed 1 (1/1000) ...
Generating image for seed 2 (2/1000) ...
Generating image for seed 3 (3/1000) ...
Generating image for seed 4 (4/1000) ...
Generating image for seed 5 (5/1000) ...
Generating image for seed 6 (6/1000) ...
Generating image for seed 7 (7/1000) ...
Generating image fo

In [19]:
# import shutil
# from google.colab import files

# shutil.make_archive('/content/normal_images', 'zip', '/content/generated/sg2ada_cond_normal')
# shutil.make_archive('/content/abnormal_images', 'zip', '/content/generated/sg2ada_cond_abnormal')

'/content/abnormal_images.zip'

In [21]:
# files.download('/content/normal_images.zip')
# files.download('/content/abnormal_images.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Evaluation

- **FID (Fréchet Inception Distance):** evaluates the overall quality and realism of your synthetic retinal images. A low FID score means the statistical distribution of features (like color, texture, and structural layout) in generated images closely matches the distribution of the real RFMiD dataset.

- **SSIM (Structural Similarity Index Measure):** calculates the structural, luminance, and contrast similarity between paired images.

- **PSNR (Peak Signal-to-Noise Ratio):** measures the ratio between the maximum possible power of a signal (the clean image data) and the corrupting noise affecting its representation.

In [3]:
def count_images(d):
    return len(glob.glob(os.path.join(d, "*.png")) +
               glob.glob(os.path.join(d, "*.jpg")))

def copy_n_sorted(src_dir, dst_dir, n, prefix, seed=42):
    """
    Copy exactly n images from src_dir into dst_dir.
    Named with prefix + index so sort order is guaranteed.
    """
    paths = sorted(
        glob.glob(os.path.join(src_dir, "*.png")) +
        glob.glob(os.path.join(src_dir, "*.jpg"))
    )
    random.seed(seed)
    selected = random.sample(paths, min(n, len(paths)))
    # Re-sort after sampling so filenames are deterministic
    selected = sorted(selected)

    for i, p in enumerate(selected):
        ext = os.path.splitext(p)[1]
        # prefix ensures normal comes before abnormal in sort order
        shutil.copy2(p, os.path.join(dst_dir, f"{prefix}_{i:05d}{ext}"))

    print(f"  Copied {len(selected)} {prefix} → {dst_dir}")
    return len(selected)

def calculate_metrics(real_dir, fake_dir, device, n_samples,
                      batch_size=32, seed=42, preserve_order=False):
    # FID
    try:
        fid = fid_score.calculate_fid_given_paths(
            [real_dir, fake_dir],
            batch_size=batch_size,
            device=device,
            dims=2048,
        )
    except Exception as e:
        print(f"  FID error: {e}")
        fid = None

    # SSIM & PSNR
    real_paths = sorted(
        glob.glob(os.path.join(real_dir, "*.png")) +
        glob.glob(os.path.join(real_dir, "*.jpg"))
    )
    fake_paths = sorted(
        glob.glob(os.path.join(fake_dir, "*.png")) +
        glob.glob(os.path.join(fake_dir, "*.jpg"))
    )

    n_pairs = min(len(real_paths), len(fake_paths), n_samples)

    if preserve_order:
        # Keep positional pairing
        # Just truncate to n_pairs, no shuffling
        real_paths = real_paths[:n_pairs]
        fake_paths = fake_paths[:n_pairs]
    else:
        # Original random sampling — for when pairing doesn't matter
        random.seed(seed)
        real_paths = random.sample(real_paths, n_pairs)
        fake_paths = random.sample(fake_paths, n_pairs)

    ssim_scores = []
    psnr_scores = []
    skipped     = 0

    for real_path, fake_path in zip(real_paths, fake_paths):
        real_img = cv2.imread(real_path, cv2.IMREAD_GRAYSCALE)
        fake_img = cv2.imread(fake_path, cv2.IMREAD_GRAYSCALE)

        if real_img is None or fake_img is None:
            skipped += 1
            continue

        # Resize real DOWN to 128 to match GAN output
        if real_img.shape != (GAN_RES, GAN_RES):
            real_img = cv2.resize(
                real_img,
                (GAN_RES, GAN_RES),
                interpolation=cv2.INTER_LANCZOS4,
            )

        # Safety check on gen size
        if fake_img.shape != (GAN_RES, GAN_RES):
            fake_img = cv2.resize(
                fake_img,
                (GAN_RES, GAN_RES),
                interpolation=cv2.INTER_LANCZOS4,
            )

        assert real_img.shape == fake_img.shape, \
            f"Shape mismatch: {real_img.shape} vs {fake_img.shape}"

        ssim_scores.append(ssim(real_img, fake_img, data_range=255))
        psnr_scores.append(psnr(real_img, fake_img, data_range=255))

    if skipped > 0:
        print(f"  Skipped {skipped} pairs due to read errors")

    avg_ssim = float(np.mean(ssim_scores)) if ssim_scores else -1.0
    avg_psnr = float(np.mean(psnr_scores)) if psnr_scores else -1.0

    return fid, avg_ssim, avg_psnr

In [4]:
import numpy as np
import glob, os, shutil, random
import cv2
import torch
import json

import subprocess
subprocess.run("pip install -q pytorch-fid", shell=True, check=True)

from pytorch_fid import fid_score
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from pathlib import Path

In [8]:
ZIP_NORMAL   = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/sg2ada_normal.zip"
ZIP_ABNORMAL = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/sg2ada_abnormal.zip"
ZIP_EVAL     = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/eval_img.zip"

GEN_NORMAL_DIR   = "/content/eval/gen_normal"
GEN_ABNORMAL_DIR = "/content/eval/gen_abnormal"
EVAL_REAL_DIR    = "/content/eval/real"
EVAL_IMG_DIR    = "/content/eval/real/images"
COMBINED_GEN_DIR = "/content/eval/combined_gen"

GAN_RES = 256
SEED    = 42
N_EACH  = 500
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
for d in [GEN_NORMAL_DIR, GEN_ABNORMAL_DIR, EVAL_REAL_DIR, COMBINED_GEN_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

print("Unzipping...")
shutil.unpack_archive(ZIP_NORMAL,   GEN_NORMAL_DIR)
shutil.unpack_archive(ZIP_ABNORMAL, GEN_ABNORMAL_DIR)
shutil.unpack_archive(ZIP_EVAL,     EVAL_REAL_DIR)

print(f"  gen normal      : {count_images(GEN_NORMAL_DIR)}")
print(f"  gen abnormal    : {count_images(GEN_ABNORMAL_DIR)}")
print(f"  eval real total : {count_images(EVAL_IMG_DIR)}")

Unzipping...
  gen normal      : 1000
  gen abnormal    : 1000
  eval real total : 1000


In [10]:
print("\nBuilding combined_gen (normal first, abnormal second)...")
n_gen_normal   = copy_n_sorted(GEN_NORMAL_DIR,   COMBINED_GEN_DIR,
                                N_EACH, "a_normal",   SEED)
n_gen_abnormal = copy_n_sorted(GEN_ABNORMAL_DIR, COMBINED_GEN_DIR,
                                N_EACH, "b_abnormal", SEED)

print(f"  combined_gen total : {count_images(COMBINED_GEN_DIR)}")


Building combined_gen (normal first, abnormal second)...
  Copied 500 a_normal → /content/eval/combined_gen
  Copied 500 b_abnormal → /content/eval/combined_gen
  combined_gen total : 1000


In [11]:
real_paths = sorted(
    glob.glob(os.path.join(EVAL_IMG_DIR,    "*.png")) +
    glob.glob(os.path.join(EVAL_IMG_DIR,    "*.jpg"))
)
gen_paths  = sorted(
    glob.glob(os.path.join(COMBINED_GEN_DIR, "*.png")) +
    glob.glob(os.path.join(COMBINED_GEN_DIR, "*.jpg"))
)

assert len(real_paths) == len(gen_paths), \
    f"Count mismatch: {len(real_paths)} real vs {len(gen_paths)} gen"

print(f"Pairs ready: {len(real_paths)} ✓")

Pairs ready: 1000 ✓


In [12]:
print("Computing metrics...")
fid_val, avg_ssim, avg_psnr = calculate_metrics(
    real_dir       = EVAL_IMG_DIR,
    fake_dir       = COMBINED_GEN_DIR,
    device         = DEVICE,
    n_samples      = N_EACH * 2,    # 1000 total pairs
    batch_size     = 32,
    seed           = SEED,
    preserve_order = True,          # ← keeps curated order
)

Computing metrics...
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 607MB/s]
100%|██████████| 32/32 [00:03<00:00,  8.92it/s]


In [13]:
print(f"""
┌──────────────────────────┬──────────────┬──────────────┬──────────────┐
│ Evaluation               │     FID ↓    │    SSIM ↑    │    PSNR ↑    │
├──────────────────────────┼──────────────┼──────────────┼──────────────┤
│ Combined (1000 pairs)    │ {fid_val:12.4f} │ {avg_ssim:12.4f} │ {avg_psnr:12.4f} │
└──────────────────────────┴──────────────┴──────────────┴──────────────┘
FID  : lower is better  — 1000 real vs 1000 generated
SSIM : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
PSNR : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
""")


┌──────────────────────────┬──────────────┬──────────────┬──────────────┐
│ Evaluation               │     FID ↓    │    SSIM ↑    │    PSNR ↑    │
├──────────────────────────┼──────────────┼──────────────┼──────────────┤
│ Combined (1000 pairs)    │      27.6960 │       0.6101 │      17.8814 │
└──────────────────────────┴──────────────┴──────────────┴──────────────┘
FID  : lower is better  — 1000 real vs 1000 generated
SSIM : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
PSNR : higher is better — 1000 positional pairs (500 normal + 500 abnormal)



Discussion
- From the evaluation result, it shows that StyleGAN2-ADA establishes an FID of 27.70, SSIM of 0.61, and PSNR of 17.88.
- By operating at 256×256 resolution, the model successfully resolved complex, high-frequency details, generating sharp vascular networks and realistic background pigmentation that previously appeared muddy.
- Ultimately, these exceptional results validate the power of Adaptive Discriminator Augmentation (ADA) by dynamically preventing the discriminator from overfitting on a limited medical dataset, ADA successfully forced the generator to learn true biological structures without memorizing the training data or succumbing to mode collapse.